# 1. The Baïou-Balinski formulation(SO-BB): 

In [ ]:
import pandas as pd
import pyomo.environ as pyo
import time  # Added to measure execution time


# 1. Initial Settings (Slicer)
NUM_STUDENTS = 1500 

COLLEGES_FILE = 'strict_colleges.csv'
APPLICATIONS_FILE = 'strict_applications.csv'

# 2. Loading and Data Preparation
print("Loading and slicing data...")

df_colleges = pd.read_csv(COLLEGES_FILE)
df_apps = pd.read_csv(APPLICATIONS_FILE)

unique_students = df_apps['student_id'].unique()[:NUM_STUDENTS]
df_apps_sliced = df_apps[df_apps['student_id'].isin(unique_students)]

active_colleges = df_apps_sliced['college_id'].unique()
df_colleges_sliced = df_colleges[df_colleges['college_id'].isin(active_colleges)]

capacity_dict = df_colleges_sliced.set_index('college_id')['capacity'].to_dict()

rank_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['rank'].to_dict()
score_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['score'].to_dict()

E_list = list(rank_dict.keys())

student_choices = {i: [] for i in unique_students}
college_applicants = {j: [] for j in active_colleges}
for (i, j) in E_list:
    student_choices[i].append(j)
    college_applicants[j].append(i)

print(f"Data ready: {len(unique_students)} Students, {len(active_colleges)} Colleges, {len(E_list)} Applications.")

# 3. Building the Mathematical Model in Pyomo (SO-BB Model)
print("\nBuilding the Pyomo model (SO-BB)...")
model = pyo.ConcreteModel(name="Student_Optimal_Baiou_Balinski")

model.A = pyo.Set(initialize=unique_students)
model.C = pyo.Set(initialize=active_colleges)
model.E = pyo.Set(initialize=E_list, dimen=2)

model.u = pyo.Param(model.C, initialize=capacity_dict)
model.r = pyo.Param(model.E, initialize=rank_dict)
model.s = pyo.Param(model.E, initialize=score_dict)

model.x = pyo.Var(model.E, domain=pyo.Binary)

# Objective Function (Eq 4): Minimize the sum of accepted ranks
def obj_rule(m):
    return sum(m.r[i, j] * m.x[i, j] for i, j in m.E)
model.Objective = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

# Constraint (1): Each applicant is accepted to at most one college
def student_capacity_rule(m, i):
    # If the student has no choices, skip the constraint
    if not student_choices[i]:
        return pyo.Constraint.Skip
    return sum(m.x[i, j] for j in student_choices[i]) <= 1
model.Constraint1 = pyo.Constraint(model.A, rule=student_capacity_rule)

# Constraint (2): Respect the capacity limit of each college
def college_capacity_rule(m, j):
    if not college_applicants[j]:
        return pyo.Constraint.Skip
    return sum(m.x[i, j] for i in college_applicants[j]) <= m.u[j]
model.Constraint2 = pyo.Constraint(model.C, rule=college_capacity_rule)

# Constraint (3): Baïou-Balinski Stability
def stability_rule(m, i, j):
    # Part 1: Is the student accepted in this college or a better one?
    better_or_equal_choices = [k for k in student_choices[i] if m.r[i, k] <= m.r[i, j]]
    term1 = sum(m.x[i, k] for k in better_or_equal_choices) * m.u[j]
    
    # Part 2: Total number of accepted students in college j with a score higher than i
    better_applicants = [h for h in college_applicants[j] if m.s[h, j] > m.s[i, j]]
    term2 = sum(m.x[h, j] for h in better_applicants)
    
    # The sum must be greater than or equal to the college capacity
    return term1 + term2 >= m.u[j]
model.Constraint3 = pyo.Constraint(model.E, rule=stability_rule)

# 4. Solving the Model, Measuring Time, and Extracting Results
print("Solving the model with GLPK...")

start_time = time.time()  
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)
end_time = time.time()    

solve_time = end_time - start_time

if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print(f"\nOptimal Solution Found in {solve_time:.4f} seconds")
    
    assignments = []
    assignment_tuples = [] # Store (student, college) pairs for comparison
    
    for (i, j) in model.E:
        if pyo.value(model.x[i, j]) > 0.5:
            assignments.append({
                'student_id': i, 
                'college_id': j, 
                'rank': rank_dict[i,j], 
                'score': score_dict[i,j]
            })
            assignment_tuples.append((i, j))
            
    df_results = pd.DataFrame(assignments)
    
    # Sort by student ID for easier comparison
    df_results = df_results.sort_values('student_id').reset_index(drop=True)
    assignment_tuples.sort() # Sort tuples to generate a consistent hash
    
    print(f"Total Students Assigned: {len(df_results)} out of {len(unique_students)}")
    print(f"Solver Time: {solve_time:.4f} seconds")
    print(f"Solution Signature (Hash): {hash(tuple(assignment_tuples))}")
    
    print("\nFirst 10 Assignments:")
    print(df_results.head(10))
    
else:
    print(f"\nSolver did not find an optimal solution. Status: {results.solver.termination_condition}")

Loading and slicing data...
Data ready: 1500 Students, 20 Colleges, 11184 Applications.

Building the Pyomo model (SO-BB)...
Solving the model with GLPK...

Optimal Solution Found in 62.1637 seconds
Total Students Assigned: 1500 out of 1500
Solver Time: 62.1637 seconds
Solution Signature (Hash): 2193633351276812154

First 10 Assignments:
   student_id  college_id  rank       score
0           1           8     1  497.998666
1           2          10     1  351.234156
2           3           4     1  420.947298
3           4          17     1  211.474316
4           5          19     1  181.120747
5           6           8     1  194.129420
6           7          16     1    7.004670
7           8           8     1  440.960640
8           9          18     1  283.522348
9          10           3     1  440.293529


# 2. (SO-NW-CUT) The cutoff score formulation: 

In [ ]:
import pandas as pd
import pyomo.environ as pyo
import time  

# 1. Initial Settings (Slicer)
NUM_STUDENTS = 1500 

COLLEGES_FILE = 'strict_colleges.csv'
APPLICATIONS_FILE = 'strict_applications.csv'

# 2. Loading and Data Preparation
print("Loading and slicing data...")

df_colleges = pd.read_csv(COLLEGES_FILE)
df_apps = pd.read_csv(APPLICATIONS_FILE)

# Apply Slicer
unique_students = df_apps['student_id'].unique()[:NUM_STUDENTS]
df_apps_sliced = df_apps[df_apps['student_id'].isin(unique_students)]

active_colleges = df_apps_sliced['college_id'].unique()
df_colleges_sliced = df_colleges[df_colleges['college_id'].isin(active_colleges)]

capacity_dict = df_colleges_sliced.set_index('college_id')['capacity'].to_dict()
rank_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['rank'].to_dict()
score_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['score'].to_dict()

E_list = list(rank_dict.keys())

student_choices = {i: [] for i in unique_students}
college_applicants = {j: [] for j in active_colleges}
for (i, j) in E_list:
    student_choices[i].append(j)
    college_applicants[j].append(i)

#New parameters needed for this model
S_BAR = max(score_dict.values()) # The highest possible score in the system
EPSILON = 0.0001             

print(f"Data ready: {len(unique_students)} Students, {len(active_colleges)} Colleges, {len(E_list)} Applications.")
print(f"S_BAR (Max Score) dynamically set to: {S_BAR:.4f}")

# 3. Building the Mathematical Model in Pyomo (SO-NW-CUT)
print("\nBuilding the Pyomo model (SO-NW-CUT)...")
model = pyo.ConcreteModel(name="Student_Optimal_Cutoff_Formulation")

# Sets and Parameters
model.A = pyo.Set(initialize=unique_students)
model.C = pyo.Set(initialize=active_colleges)
model.E = pyo.Set(initialize=E_list, dimen=2)

model.u = pyo.Param(model.C, initialize=capacity_dict)
model.r = pyo.Param(model.E, initialize=rank_dict)
model.s = pyo.Param(model.E, initialize=score_dict)

# Decision Variables
model.x = pyo.Var(model.E, domain=pyo.Binary)                 # Student assignment
model.t = pyo.Var(model.C, domain=pyo.NonNegativeReals)       # Continuous cutoff score
model.f = pyo.Var(model.C, domain=pyo.Binary)                 # Indicator if college rejected any applicant (college is full)

# Objective Function (Eq 4): Minimize the sum of accepted ranks
def obj_rule(m):
    return sum(m.r[i, j] * m.x[i, j] for i, j in m.E)
model.Objective = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

# Constraint (1): Student Capacity
def student_capacity_rule(m, i):
    if not student_choices[i]: return pyo.Constraint.Skip
    return sum(m.x[i, j] for j in student_choices[i]) <= 1
model.Constraint1 = pyo.Constraint(model.A, rule=student_capacity_rule)

# Constraint (2): College Capacity
def college_capacity_rule(m, j):
    if not college_applicants[j]: return pyo.Constraint.Skip
    return sum(m.x[i, j] for i in college_applicants[j]) <= m.u[j]
model.Constraint2 = pyo.Constraint(model.C, rule=college_capacity_rule)

# Constraint (5): Admissibility based on cutoff score
def admissibility_rule(m, i, j):
    # t_j <= (1 - x_ij)*(S_BAR + 1) + s_ij
    return m.t[j] <= (1 - m.x[i, j]) * (S_BAR + 1) + m.s[i, j]
model.Constraint5 = pyo.Constraint(model.E, rule=admissibility_rule)

# Constraint (6): Envy-freeness
def envy_free_rule(m, i, j):
    better_or_equal_choices = [k for k in student_choices[i] if m.r[i, k] <= m.r[i, j]]
    sum_better_x = sum(m.x[i, k] for k in better_or_equal_choices)
    
    # s_ij + EPSILON <= t_j + sum_better * (S_BAR + 1)
    return m.s[i, j] + EPSILON <= m.t[j] + sum_better_x * (S_BAR + 1)
model.Constraint6 = pyo.Constraint(model.E, rule=envy_free_rule)

# Constraint (7): Non-wastefulness (Part 1 - Must be full to reject)
def non_wasteful_full_rule(m, j):
    if not college_applicants[j]: return pyo.Constraint.Skip
    # f_j * u_j <= sum(x_ij)
    return m.f[j] * m.u[j] <= sum(m.x[i, j] for i in college_applicants[j])
model.Constraint7 = pyo.Constraint(model.C, rule=non_wasteful_full_rule)

# Constraint (8): Non-wastefulness (Part 2 - Zero Cutoff if not full)
def non_wasteful_cutoff_rule(m, j):
    # t_j <= f_j * (S_BAR + 1)
    return m.t[j] <= m.f[j] * (S_BAR + 1)
model.Constraint8 = pyo.Constraint(model.C, rule=non_wasteful_cutoff_rule)

# 4. Solving the Model, Measuring Time, and Extracting Results
print("Solving the model with GLPK...")

start_time = time.time()  # Record solve start time
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)
end_time = time.time()    # Record solve end time

solve_time = end_time - start_time

if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print(f"\nOptimal Solution Found in {solve_time:.4f} seconds")
    
    assignments = []
    assignment_tuples = [] # Store (student, college) pairs for comparison
    
    for (i, j) in model.E:
        if pyo.value(model.x[i, j]) > 0.5:
            assignments.append({
                'student_id': i, 
                'college_id': j, 
                'rank': rank_dict[i,j], 
                'score': score_dict[i,j]
            })
            assignment_tuples.append((i, j))
            
    df_results = pd.DataFrame(assignments)
    
    # Sort by student ID for easier comparison
    df_results = df_results.sort_values('student_id').reset_index(drop=True)
    assignment_tuples.sort() # Sort tuples to generate a consistent hash
    
    print(f"Total Students Assigned: {len(df_results)} out of {len(unique_students)}")
    print(f"Solver Time: {solve_time:.4f} seconds")
    print(f"Solution Signature (Hash): {hash(tuple(assignment_tuples))}")
    
    # Print the calculated cutoff scores for colleges
    print("\nCollege Cutoff Scores (t_j):")
    for j in active_colleges:
        if pyo.value(model.t[j]) > 0: # Print only colleges with a non-zero cutoff
            print(f"College {j:2} | Cutoff: {pyo.value(model.t[j]):.4f} | Full (f_j): {pyo.value(model.f[j]):.0f}")
            
    print("\nFirst 10 Assignments:")
    print(df_results.head(10))
    
else:
    print(f"\nSolver did not find an optimal solution. Status: {results.solver.termination_condition}")

Loading and slicing data...
Data ready: 1500 Students, 20 Colleges, 11184 Applications.
S_BAR (Max Score) dynamically set to: 500.0000

Building the Pyomo model (SO-NW-CUT)...
Solving the model with GLPK...

Optimal Solution Found in 11.7047 seconds
Total Students Assigned: 1500 out of 1500
Solver Time: 11.7047 seconds
Solution Signature (Hash): 2193633351276812154

College Cutoff Scores (t_j):
College  8 | Cutoff: 118.7459 | Full (f_j): 1
College  3 | Cutoff: 83.7226 | Full (f_j): 1
College  9 | Cutoff: 32.3550 | Full (f_j): 1
College 17 | Cutoff: 87.7253 | Full (f_j): 1
College 18 | Cutoff: 214.4764 | Full (f_j): 1
College 11 | Cutoff: 187.1247 | Full (f_j): 1
College  2 | Cutoff: 145.7639 | Full (f_j): 1
College  4 | Cutoff: 11.3410 | Full (f_j): 1
College 12 | Cutoff: 6.6712 | Full (f_j): 1
College  1 | Cutoff: 77.3850 | Full (f_j): 1
College 10 | Cutoff: 140.4271 | Full (f_j): 1
College  7 | Cutoff: 239.8266 | Full (f_j): 1
College 15 | Cutoff: 22.3483 | Full (f_j): 1
College 13 |

# 3. (SO-NW-BIN-CUT) The binary cutoff score formulation: 

In [ ]:
import pandas as pd
import pyomo.environ as pyo
import time


# 1. Initial Settings (Slicer)

NUM_STUDENTS = 1500 

COLLEGES_FILE = 'strict_colleges.csv'
APPLICATIONS_FILE = 'strict_applications.csv'


# 2. Loading and Data Preparation

print("Loading and slicing data...")

df_colleges = pd.read_csv(COLLEGES_FILE)
df_apps = pd.read_csv(APPLICATIONS_FILE)

unique_students = df_apps['student_id'].unique()[:NUM_STUDENTS]
df_apps_sliced = df_apps[df_apps['student_id'].isin(unique_students)]

active_colleges = df_apps_sliced['college_id'].unique()
df_colleges_sliced = df_colleges[df_colleges['college_id'].isin(active_colleges)]

capacity_dict = df_colleges_sliced.set_index('college_id')['capacity'].to_dict()
rank_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['rank'].to_dict()
score_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['score'].to_dict()

E_list = list(rank_dict.keys())

student_choices = {i: [] for i in unique_students}
college_applicants = {j: [] for j in active_colleges}
for (i, j) in E_list:
    student_choices[i].append(j)
    college_applicants[j].append(i)

# --- Extract the set S_j (distinct and sorted scores for each college) ---
S_j_dict = {}       
T_indices = []      
for j in active_colleges:
    # Collect all scores of applicants for this college, remove duplicates, and sort ascending
    scores_for_j = sorted(list(set(score_dict[i, j] for i in college_applicants[j])))
    S_j_dict[j] = scores_for_j
    
    # Create valid indices for the binary variable t_j^k
    for s in scores_for_j:
        T_indices.append((j, s))

print(f"Data ready: {len(unique_students)} Students, {len(active_colleges)} Colleges, {len(E_list)} Applications.")
print(f"Total Binary Cutoff Variables (t_j^k) to create: {len(T_indices)}")


# 3. Building the Pyomo Model (SO-NW-BIN-CUT)

print("\nBuilding the Pyomo model (SO-NW-BIN-CUT)...")
model = pyo.ConcreteModel(name="Student_Optimal_Binary_Cutoff")

# Sets and Parameters
model.A = pyo.Set(initialize=unique_students)
model.C = pyo.Set(initialize=active_colleges)
model.E = pyo.Set(initialize=E_list, dimen=2)
model.T_set = pyo.Set(initialize=T_indices, dimen=2) # Index set for binary cutoffs (j, score)

model.u = pyo.Param(model.C, initialize=capacity_dict)
model.r = pyo.Param(model.E, initialize=rank_dict)
model.s = pyo.Param(model.E, initialize=score_dict)

# Decision Variables
model.x = pyo.Var(model.E, domain=pyo.Binary)     # Student assignment
model.t = pyo.Var(model.T_set, domain=pyo.Binary) # Binary cutoff variable


# Objective Function (Eq 4): Minimize the sum of accepted ranks

def obj_rule(m):
    return sum(m.r[i, j] * m.x[i, j] for i, j in m.E)
model.Objective = pyo.Objective(rule=obj_rule, sense=pyo.minimize)


# Constraint (1): Student Capacity

def student_capacity_rule(m, i):
    if not student_choices[i]: return pyo.Constraint.Skip
    return sum(m.x[i, j] for j in student_choices[i]) <= 1
model.Constraint1 = pyo.Constraint(model.A, rule=student_capacity_rule)


# Constraint (2): College Capacity

def college_capacity_rule(m, j):
    if not college_applicants[j]: return pyo.Constraint.Skip
    return sum(m.x[i, j] for i in college_applicants[j]) <= m.u[j]
model.Constraint2 = pyo.Constraint(model.C, rule=college_capacity_rule)


# Constraint (11): Admissibility based on binary cutoff variable

def binary_admissibility_rule(m, i, j):
    s_val = m.s[i, j]
    return m.x[i, j] <= m.t[j, s_val]
model.Constraint11 = pyo.Constraint(model.E, rule=binary_admissibility_rule)

# Constraint (12): Monotonicity of cutoff variables
def monotonicity_rule(m, j):
    constraints = []
    scores = S_j_dict[j]
    for k in range(len(scores) - 1):
        s_current = scores[k]
        s_next = scores[k+1]
        constraints.append(m.t[j, s_current] <= m.t[j, s_next])
    return constraints
model.Constraint12 = pyo.ConstraintList()
for j in active_colleges:
    for expr in monotonicity_rule(model, j):
        model.Constraint12.add(expr)

# Constraint (13): Binary Envy-freeness (No Big-M)
def binary_envy_free_rule(m, i, j):
    s_val = m.s[i, j]
    better_or_equal_choices = [k for k in student_choices[i] if m.r[i, k] <= m.r[i, j]]
    sum_better_x = sum(m.x[i, k] for k in better_or_equal_choices)
    
    return 1 <= sum_better_x + (1 - m.t[j, s_val])
model.Constraint13 = pyo.Constraint(model.E, rule=binary_envy_free_rule)

# Constraint (14): Binary Non-wastefulness
def binary_non_wasteful_rule(m, j):
    if not college_applicants[j]: return pyo.Constraint.Skip
    
    lowest_score = S_j_dict[j][0]
    return (1 - m.t[j, lowest_score]) * m.u[j] <= sum(m.x[i, j] for i in college_applicants[j])
model.Constraint14 = pyo.Constraint(model.C, rule=binary_non_wasteful_rule)

# 4. Solving the Model, Measuring Time, and Extracting Results
print("Solving the model with GLPK...")

start_time = time.time()
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)
end_time = time.time()

solve_time = end_time - start_time

if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print(f"\nOptimal Solution Found in {solve_time:.4f} seconds")
    
    assignments = []
    assignment_tuples = [] 
    
    for (i, j) in model.E:
        if pyo.value(model.x[i, j]) > 0.5:
            assignments.append({
                'student_id': i, 
                'college_id': j, 
                'rank': rank_dict[i,j], 
                'score': score_dict[i,j]
            })
            assignment_tuples.append((i, j))
            
    df_results = pd.DataFrame(assignments)
    
    # Sort for easier comparison
    df_results = df_results.sort_values('student_id').reset_index(drop=True)
    assignment_tuples.sort() 
    
    print(f"Total Students Assigned: {len(df_results)} out of {len(unique_students)}")
    print(f"Solver Time: {solve_time:.4f} seconds")
    print(f"Solution Signature (Hash): {hash(tuple(assignment_tuples))}")
    
    # --- Extract the final cutoff score for each college from the binary variables ---
    print("\nFinal Cutoff Scores for each College:")
    for j in active_colleges:
        final_cutoff = None
        # The first score for which t_j^k equals 1 is the final cutoff
        for s in S_j_dict[j]:
            if pyo.value(model.t[j, s]) > 0.5:
                final_cutoff = s
                break
        
        if final_cutoff is not None:
            print(f"College {j:2} | Cutoff Score: {final_cutoff:.4f}")
        else:
            print(f"College {j:2} | No Cutoff (All rejected or 0 applicants)")
            
    print("\nFirst 10 Assignments:")
    print(df_results.head(10))
    
else:
    print(f"\nSolver did not find an optimal solution. Status: {results.solver.termination_condition}")

Loading and slicing data...
Data ready: 1500 Students, 20 Colleges, 11184 Applications.
Total Binary Cutoff Variables (t_j^k) to create: 11184

Building the Pyomo model (SO-NW-BIN-CUT)...
Solving the model with GLPK...

Optimal Solution Found in 13.1179 seconds
Total Students Assigned: 1500 out of 1500
Solver Time: 13.1179 seconds
Solution Signature (Hash): 2193633351276812154

Final Cutoff Scores for each College:
College  8 | Cutoff Score: 119.4129
College  3 | Cutoff Score: 93.3956
College  9 | Cutoff Score: 35.3569
College 17 | Cutoff Score: 91.3943
College 18 | Cutoff Score: 222.4817
College 11 | Cutoff Score: 187.1247
College  2 | Cutoff Score: 154.1027
College  4 | Cutoff Score: 15.0100
College 12 | Cutoff Score: 8.0053
College  1 | Cutoff Score: 80.7205
College 10 | Cutoff Score: 157.4383
College  7 | Cutoff Score: 239.8266
College 15 | Cutoff Score: 24.6831
College 13 | Cutoff Score: 187.7919
College  5 | Cutoff Score: 68.0454
College 14 | Cutoff Score: 0.3336
College 19 | Cut

# 4. (MSMR-EF) Envy-free formulation: 

In [ ]:
import pandas as pd
import pyomo.environ as pyo
import time


# 1. Initial Settings (Slicer)

NUM_STUDENTS = 1500 

COLLEGES_FILE = 'strict_colleges.csv'
APPLICATIONS_FILE = 'strict_applications.csv'


# 2. Loading and Data Preparation

print("Loading and slicing data...")

df_colleges = pd.read_csv(COLLEGES_FILE)
df_apps = pd.read_csv(APPLICATIONS_FILE)

unique_students = df_apps['student_id'].unique()[:NUM_STUDENTS]
df_apps_sliced = df_apps[df_apps['student_id'].isin(unique_students)]

active_colleges = df_apps_sliced['college_id'].unique()
df_colleges_sliced = df_colleges[df_colleges['college_id'].isin(active_colleges)]

capacity_dict = df_colleges_sliced.set_index('college_id')['capacity'].to_dict()
rank_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['rank'].to_dict()
score_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['score'].to_dict()

E_list = list(rank_dict.keys())

student_choices = {i: [] for i in unique_students}
college_applicants = {j: [] for j in active_colleges}
for (i, j) in E_list:
    student_choices[i].append(j)
    college_applicants[j].append(i)

# --- Generate pairwise comparison list for Constraint (16) ---
# Find all pairs (i, h) in each college j where score i >= score h
EF_pairs = []
for j in active_colleges:
    applicants = college_applicants[j]
    for i in applicants:
        for h in applicants:
            if i != h and score_dict[i, j] >= score_dict[h, j]:
                EF_pairs.append((j, i, h))

# Constant K for the objective function
K_VAL = max(rank_dict.values()) + 1

print(f"Data ready: {len(unique_students)} Students, {len(active_colleges)} Colleges, {len(E_list)} Applications.")
print(f"Created {len(EF_pairs)} pairwise comparisons for Envy-Freeness.")

# 3. Building the Pyomo Model (MSMR-EF)
print("\nBuilding the Pyomo model (MSMR-EF)...")
model = pyo.ConcreteModel(name="Max_Size_Min_Rank_Envy_Free")

# Sets and Parameters
model.A = pyo.Set(initialize=unique_students)
model.C = pyo.Set(initialize=active_colleges)
model.E = pyo.Set(initialize=E_list, dimen=2)
model.EF_Set = pyo.Set(initialize=EF_pairs, dimen=3) # Set (j, i, h)

model.u = pyo.Param(model.C, initialize=capacity_dict)
model.r = pyo.Param(model.E, initialize=rank_dict)
model.s = pyo.Param(model.E, initialize=score_dict)

# Decision Variable (Only x exists!)
model.x = pyo.Var(model.E, domain=pyo.Binary)

# Objective Function (Eq 10): Maximize assignments with minimum rank
def obj_rule(m):
    return sum((K_VAL - m.r[i, j]) * m.x[i, j] for i, j in m.E)
model.Objective = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

# Constraint (1): Student Capacity
def student_capacity_rule(m, i):
    if not student_choices[i]: return pyo.Constraint.Skip
    return sum(m.x[i, j] for j in student_choices[i]) <= 1
model.Constraint1 = pyo.Constraint(model.A, rule=student_capacity_rule)

# Constraint (2): College Capacity
def college_capacity_rule(m, j):
    if not college_applicants[j]: return pyo.Constraint.Skip
    return sum(m.x[i, j] for i in college_applicants[j]) <= m.u[j]
model.Constraint2 = pyo.Constraint(model.C, rule=college_capacity_rule)

# Constraint (16): Direct Envy-Freeness (Pairwise)
def direct_envy_free_rule(m, j, i, h):
    # Sum of student i's assignments in choices equal to or better than j
    better_or_equal_choices = [k for k in student_choices[i] if m.r[i, k] <= m.r[i, j]]
    lhs = sum(m.x[i, k] for k in better_or_equal_choices)
    
    # Must be greater than or equal to the acceptance status of the weaker student (h)
    rhs = m.x[h, j]
    return lhs >= rhs
model.Constraint16 = pyo.Constraint(model.EF_Set, rule=direct_envy_free_rule)

# 4. Solving the Model, Measuring Time, and Extracting Results
print("Solving the model with GLPK...")

start_time = time.time()
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model, tee=True)
end_time = time.time()

solve_time = end_time - start_time

if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print(f"\nOptimal Solution Found in {solve_time:.4f} seconds")
    
    assignments = []
    assignment_tuples = [] 
    
    for (i, j) in model.E:
        if pyo.value(model.x[i, j]) > 0.5:
            assignments.append({
                'student_id': i, 
                'college_id': j, 
                'rank': rank_dict[i,j], 
                'score': score_dict[i,j]
            })
            assignment_tuples.append((i, j))
            
    df_results = pd.DataFrame(assignments)
    
    # Sort by student ID for easier comparison
    df_results = df_results.sort_values('student_id').reset_index(drop=True)
    assignment_tuples.sort() 
    
    print(f"Total Students Assigned: {len(df_results)} out of {len(unique_students)}")
    print(f"Solver Time: {solve_time:.4f} seconds")
    print(f"Solution Signature (Hash): {hash(tuple(assignment_tuples))}")
            
    print("\nFirst 10 Assignments:")
    print(df_results.head(10))
    
else:
    print(f"\nSolver did not find an optimal solution. Status: {results.solver.termination_condition}")

Loading and slicing data...
Data ready: 1500 Students, 20 Colleges, 11184 Applications.
Created 3125799 pairwise comparisons for Envy-Freeness.

Building the Pyomo model (MSMR-EF)...
Solving the model with GLPK...
Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
MIP has 3127319 rows; 11184 cols; 16910024 nonzeros; 11184 integer variables (11184 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 1e+01]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+02]
Presolving model
3127319 rows, 11184 cols, 16910024 nonzeros 9s
   Considered 135 / 11184 binaries; 135 probed  (rate 37/ms => expected probing finish time 485s) 74s
   Considered 272 / 11184 binaries; 270 probed  (rate 37/ms => expected probing finish time 485s) 80s
   Considered 420 / 11184 binaries; 411 probed  (rate 36/ms => expected probing finish time 479s) 85s
   Considered 566 / 11184 binaries; 551 probed  (rate 38/ms => expected probing finish time 503s) 90s
   Considered 703

# 5.  The cutoff score formulation with ties for Restrictive Policy (SO-H-NW-CUT): 

In [ ]:
import pandas as pd
import pyomo.environ as pyo
import time

# 1. Initial Settings (Slicer)
NUM_STUDENTS = 1500 

COLLEGES_FILE = 'ties_colleges.csv'
APPLICATIONS_FILE = 'ties_applications.csv'

# 2. Loading and Data Preparation
print("Loading and slicing data for Ties (Hungarian Policy)...")

df_colleges = pd.read_csv(COLLEGES_FILE)
df_apps = pd.read_csv(APPLICATIONS_FILE)

unique_students = df_apps['student_id'].unique()[:NUM_STUDENTS]
df_apps_sliced = df_apps[df_apps['student_id'].isin(unique_students)]

active_colleges = df_apps_sliced['college_id'].unique()
df_colleges_sliced = df_colleges[df_colleges['college_id'].isin(active_colleges)]

capacity_dict = df_colleges_sliced.set_index('college_id')['capacity'].to_dict()
rank_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['rank'].to_dict()
score_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['score'].to_dict()

E_list = list(rank_dict.keys())

student_choices = {i: [] for i in unique_students}
college_applicants = {j: [] for j in active_colleges}
for (i, j) in E_list:
    student_choices[i].append(j)
    college_applicants[j].append(i)

# Big-M and Epsilon parameters
S_BAR = max(score_dict.values())
EPSILON = 0.0001
# Constant K for the maximization objective function
K_VAL = max(rank_dict.values()) + 1

print(f"Data ready: {len(unique_students)} Students, {len(active_colleges)} Colleges, {len(E_list)} Applications.")
print(f"S_BAR (Max Score) dynamically set to: {S_BAR:.4f}")

# 3. Building the Pyomo Model (SO-H-NW-CUT)
print("\nBuilding the Pyomo model (SO-H-NW-CUT)...")
model = pyo.ConcreteModel(name="Hungarian_Continuous_Cutoff")

# Sets and Parameters
model.A = pyo.Set(initialize=unique_students)
model.C = pyo.Set(initialize=active_colleges)
model.E = pyo.Set(initialize=E_list, dimen=2)

model.u = pyo.Param(model.C, initialize=capacity_dict)
model.r = pyo.Param(model.E, initialize=rank_dict)
model.s = pyo.Param(model.E, initialize=score_dict)

# Decision Variables
model.x = pyo.Var(model.E, domain=pyo.Binary)                 # Student assignment
model.t = pyo.Var(model.C, domain=pyo.NonNegativeReals)       # Continuous cutoff score
model.f = pyo.Var(model.C, domain=pyo.Binary)                 # Indicator if college is full
model.d = pyo.Var(model.E, domain=pyo.Binary)                 # Auxiliary variable for marginal applicants

# Objective Function (Eq 10): Maximize assignments with minimum rank
def obj_rule(m):
    return sum((K_VAL - m.r[i, j]) * m.x[i, j] for i, j in m.E)
model.Objective = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

# Constraints 1 & 2: Student and College Capacity
def student_capacity_rule(m, i):
    if not student_choices[i]: return pyo.Constraint.Skip
    return sum(m.x[i, j] for j in student_choices[i]) <= 1
model.Constraint1 = pyo.Constraint(model.A, rule=student_capacity_rule)

def college_capacity_rule(m, j):
    if not college_applicants[j]: return pyo.Constraint.Skip
    return sum(m.x[i, j] for i in college_applicants[j]) <= m.u[j]
model.Constraint2 = pyo.Constraint(model.C, rule=college_capacity_rule)

# Constraints 5 & 6: Admissibility and Envy-freeness (from base models)
def admissibility_rule(m, i, j):
    return m.t[j] <= (1 - m.x[i, j]) * (S_BAR + 1) + m.s[i, j]
model.Constraint5 = pyo.Constraint(model.E, rule=admissibility_rule)

def envy_free_rule(m, i, j):
    better_or_equal_choices = [k for k in student_choices[i] if m.r[i, k] <= m.r[i, j]]
    sum_better_x = sum(m.x[i, k] for k in better_or_equal_choices)
    return m.s[i, j] + EPSILON <= m.t[j] + sum_better_x * (S_BAR + 1)
model.Constraint6 = pyo.Constraint(model.E, rule=envy_free_rule)

# Constraint (17): Variable d is 1 only if the applicant is not accepted to an equal or better choice
def d_preference_rule(m, i, j):
    # All choices the applicant prefers to or considers equal to j
    better_or_equal_choices = [k for k in student_choices[i] if m.r[i, k] <= m.r[i, j]]
    # If accepted to any of these (including j itself), d_ij must be zero
    return m.d[i, j] <= 1 - sum(m.x[i, k] for k in better_or_equal_choices)
model.Constraint17 = pyo.Constraint(model.E, rule=d_preference_rule)

# Constraint (18): Relationship between score and a 1-unit reduction in cutoff
def d_cutoff_rule(m, i, j):
    # t_j - 1 <= (1 - d_ij)*(S_BAR + 1) + s_ij
    return m.t[j] - 1 <= (1 - m.d[i, j]) * (S_BAR + 1) + m.s[i, j]
model.Constraint18 = pyo.Constraint(model.E, rule=d_cutoff_rule)

# Constraint (19): Non-wastefulness under the strict Hungarian policy
def hungarian_non_wasteful_rule(m, j):
    if not college_applicants[j]: return pyo.Constraint.Skip
    # f_j * (u_j + 1) <= sum(x_ij + d_ij)
    return m.f[j] * (m.u[j] + 1) <= sum(m.x[i, j] + m.d[i, j] for i in college_applicants[j])
model.Constraint19 = pyo.Constraint(model.C, rule=hungarian_non_wasteful_rule)

# Constraint (8): Zero Cutoff if college is not full
def non_wasteful_cutoff_rule(m, j):
    return m.t[j] <= m.f[j] * (S_BAR + 1)
model.Constraint8 = pyo.Constraint(model.C, rule=non_wasteful_cutoff_rule)

# 4. Solving the Model, Measuring Time, and Extracting Results
print("Solving the model with GLPK...")

start_time = time.time()
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)
end_time = time.time()

solve_time = end_time - start_time

if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print(f"\nOptimal Solution Found in {solve_time:.4f} seconds")
    
    assignments = []
    assignment_tuples = []
    
    for (i, j) in model.E:
        if pyo.value(model.x[i, j]) > 0.5:
            assignments.append({
                'student_id': i, 
                'college_id': j, 
                'rank': rank_dict[i,j], 
                'score': score_dict[i,j]
            })
            assignment_tuples.append((i, j))
            
    df_results = pd.DataFrame(assignments)
    df_results = df_results.sort_values('student_id').reset_index(drop=True)
    assignment_tuples.sort() 
    
    print(f"Total Students Assigned: {len(df_results)} out of {len(unique_students)}")
    print(f"Solver Time: {solve_time:.4f} seconds")
    print(f"Solution Signature (Hash): {hash(tuple(assignment_tuples))}")
    
    print("\nCollege Cutoff Scores (t_j):")
    for j in active_colleges:
        if pyo.value(model.t[j]) > 0:
            print(f"College {j:2} | Cutoff: {pyo.value(model.t[j]):.4f} | Full (f_j): {pyo.value(model.f[j]):.0f}")
            
    print("\nFirst 10 Assignments:")
    print(df_results.head(10))
    
    # Print a sample of applicants whose d variable is 1 (marginal applicants rejected due to ties)
    print("\nRejected Students at the Margin (d_ij = 1):")
    d_count = 0
    for (i, j) in model.E:
        if pyo.value(model.d[i, j]) > 0.5:
            print(f"Student {i:2} -> College {j:2} | Score: {score_dict[i,j]}")
            d_count += 1
    if d_count == 0: print("None found in this slice.")

else:
    print(f"\nSolver did not find an optimal solution. Status: {results.solver.termination_condition}")

Loading and slicing data for Ties (Hungarian Policy)...
Data ready: 1500 Students, 20 Colleges, 11158 Applications.
S_BAR (Max Score) dynamically set to: 500.0000

Building the Pyomo model (SO-H-NW-CUT)...
Solving the model with GLPK...

Optimal Solution Found in 79.4494 seconds
Total Students Assigned: 1493 out of 1500
Solver Time: 79.4494 seconds
Solution Signature (Hash): 52140284683300858

College Cutoff Scores (t_j):
College  4 | Cutoff: 140.0001 | Full (f_j): 1
College  3 | Cutoff: 21.0000 | Full (f_j): 1
College 14 | Cutoff: 140.0001 | Full (f_j): 1
College 20 | Cutoff: 230.0001 | Full (f_j): 1
College  9 | Cutoff: 180.0001 | Full (f_j): 1
College 17 | Cutoff: 121.0000 | Full (f_j): 1
College  5 | Cutoff: 120.0001 | Full (f_j): 1
College 12 | Cutoff: 200.0001 | Full (f_j): 1
College 11 | Cutoff: 110.0001 | Full (f_j): 1
College  2 | Cutoff: 30.0001 | Full (f_j): 1
College  6 | Cutoff: 130.0001 | Full (f_j): 1
College 15 | Cutoff: 1.0000 | Full (f_j): 1
College  1 | Cutoff: 11.00

# 6.  The binary cutoff score formulation with ties for Restrictive Policy (SO-H-NW-BIN-CUT): 

In [ ]:
import pandas as pd
import pyomo.environ as pyo
import time


# 1. Initial Settings (Slicer)

NUM_STUDENTS = 1500 

COLLEGES_FILE = 'ties_colleges.csv'
APPLICATIONS_FILE = 'ties_applications.csv'


# 2. Loading and Data Preparation

print("Loading and slicing data for Ties (Hungarian Binary Policy)...")

df_colleges = pd.read_csv(COLLEGES_FILE)
df_apps = pd.read_csv(APPLICATIONS_FILE)

unique_students = df_apps['student_id'].unique()[:NUM_STUDENTS]
df_apps_sliced = df_apps[df_apps['student_id'].isin(unique_students)]

active_colleges = df_apps_sliced['college_id'].unique()
df_colleges_sliced = df_colleges[df_colleges['college_id'].isin(active_colleges)]

capacity_dict = df_colleges_sliced.set_index('college_id')['capacity'].to_dict()
rank_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['rank'].to_dict()
score_dict = df_apps_sliced.set_index(['student_id', 'college_id'])['score'].to_dict()

E_list = list(rank_dict.keys())

student_choices = {i: [] for i in unique_students}
college_applicants = {j: [] for j in active_colleges}
for (i, j) in E_list:
    student_choices[i].append(j)
    college_applicants[j].append(i)

# --- Extract the set S_j (distinct and sorted scores for each college) ---
S_j_dict = {}       
T_indices = []      
for j in active_colleges:
    # Collect, remove duplicates, and sort ascending
    scores_for_j = sorted(list(set(score_dict[i, j] for i in college_applicants[j])))
    S_j_dict[j] = scores_for_j
    
    # Create valid indices for the binary variable t_j^k
    for s in scores_for_j:
        T_indices.append((j, s))

# Constant K for the objective function
K_VAL = max(rank_dict.values()) + 1

print(f"Data ready: {len(unique_students)} Students, {len(active_colleges)} Colleges, {len(E_list)} Applications.")
print(f"Total Binary Cutoff Variables (t_j^k) to create: {len(T_indices)}")


# 3. Building the Pyomo Model (SO-H-NW-BIN-CUT)

print("\nBuilding the Pyomo model (SO-H-NW-BIN-CUT)...")
model = pyo.ConcreteModel(name="Hungarian_Binary_Cutoff")

# Sets and Parameters
model.A = pyo.Set(initialize=unique_students)
model.C = pyo.Set(initialize=active_colleges)
model.E = pyo.Set(initialize=E_list, dimen=2)
model.T_set = pyo.Set(initialize=T_indices, dimen=2) # (j, score)

model.u = pyo.Param(model.C, initialize=capacity_dict)
model.r = pyo.Param(model.E, initialize=rank_dict)
model.s = pyo.Param(model.E, initialize=score_dict)

# Decision Variables (All Binary)
model.x = pyo.Var(model.E, domain=pyo.Binary)     # Student assignment
model.t = pyo.Var(model.T_set, domain=pyo.Binary) # Binary cutoff variable
model.d = pyo.Var(model.E, domain=pyo.Binary)     # Auxiliary marginal variable


# Objective Function (Eq 10): Maximize assignments with minimum rank

def obj_rule(m):
    return sum((K_VAL - m.r[i, j]) * m.x[i, j] for i, j in m.E)
model.Objective = pyo.Objective(rule=obj_rule, sense=pyo.maximize)


# Feasibility and Envy-freeness Constraints (from base binary models)

def student_capacity_rule(m, i):
    if not student_choices[i]: return pyo.Constraint.Skip
    return sum(m.x[i, j] for j in student_choices[i]) <= 1
model.Constraint1 = pyo.Constraint(model.A, rule=student_capacity_rule)

def college_capacity_rule(m, j):
    if not college_applicants[j]: return pyo.Constraint.Skip
    return sum(m.x[i, j] for i in college_applicants[j]) <= m.u[j]
model.Constraint2 = pyo.Constraint(model.C, rule=college_capacity_rule)

def binary_admissibility_rule(m, i, j):
    s_val = m.s[i, j]
    return m.x[i, j] <= m.t[j, s_val]
model.Constraint11 = pyo.Constraint(model.E, rule=binary_admissibility_rule)

def monotonicity_rule(m, j):
    constraints = []
    scores = S_j_dict[j]
    for k in range(len(scores) - 1):
        s_current = scores[k]
        s_next = scores[k+1]
        constraints.append(m.t[j, s_current] <= m.t[j, s_next])
    return constraints
model.Constraint12 = pyo.ConstraintList()
for j in active_colleges:
    for expr in monotonicity_rule(model, j):
        model.Constraint12.add(expr)

def binary_envy_free_rule(m, i, j):
    s_val = m.s[i, j]
    better_or_equal_choices = [k for k in student_choices[i] if m.r[i, k] <= m.r[i, j]]
    sum_better_x = sum(m.x[i, k] for k in better_or_equal_choices)
    return 1 <= sum_better_x + (1 - m.t[j, s_val])
model.Constraint13 = pyo.Constraint(model.E, rule=binary_envy_free_rule)

def d_preference_rule(m, i, j):
    better_or_equal_choices = [k for k in student_choices[i] if m.r[i, k] <= m.r[i, j]]
    return m.d[i, j] <= 1 - sum(m.x[i, k] for k in better_or_equal_choices)
model.Constraint17 = pyo.Constraint(model.E, rule=d_preference_rule)


# Constraint (20): Binary Non-wastefulness for the Hungarian Policy

def hungarian_binary_non_wasteful_rule(m, j):
    if not college_applicants[j]: return pyo.Constraint.Skip
    lowest_score = S_j_dict[j][0]
    # (1 - t_j^1) * (u_j + 1) <= sum(x_ij + d_ij)
    return (1 - m.t[j, lowest_score]) * (m.u[j] + 1) <= sum(m.x[i, j] + m.d[i, j] for i in college_applicants[j])
model.Constraint20 = pyo.Constraint(model.C, rule=hungarian_binary_non_wasteful_rule)


# Constraint (21): Binary condition for marginal applicants

def d_binary_cutoff_rule(m, i, j):
    s_val = m.s[i, j]
    scores = S_j_dict[j]
    
    # Find the position (index k) of this score in the sorted list
    k = scores.index(s_val)
    
    # If this is the highest recorded score, there is no k+1 score
    # Lowering the cutoff from above it is meaningless, so d_ij cannot be 1
    if k == len(scores) - 1:
        return m.d[i, j] <= 0
    else:
        s_next = scores[k+1] # Score of the next higher group
        # d_ij <= t_j^{k+1} - t_j^k
        return m.d[i, j] <= m.t[j, s_next] - m.t[j, s_val]
model.Constraint21 = pyo.Constraint(model.E, rule=d_binary_cutoff_rule)


# 4. Solving the Model, Measuring Time, and Extracting Results

print("Solving the model with GLPK...")

start_time = time.time()
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)
end_time = time.time()

solve_time = end_time - start_time

if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print(f"\nOptimal Solution Found in {solve_time:.4f} seconds")
    
    assignments = []
    assignment_tuples = []
    
    for (i, j) in model.E:
        if pyo.value(model.x[i, j]) > 0.5:
            assignments.append({
                'student_id': i, 
                'college_id': j, 
                'rank': rank_dict[i,j], 
                'score': score_dict[i,j]
            })
            assignment_tuples.append((i, j))
            
    df_results = pd.DataFrame(assignments)
    df_results = df_results.sort_values('student_id').reset_index(drop=True)
    assignment_tuples.sort() 
    
    print(f"Total Students Assigned: {len(df_results)} out of {len(unique_students)}")
    print(f"Solver Time: {solve_time:.4f} seconds")
    print(f"Solution Signature (Hash): {hash(tuple(assignment_tuples))}")
    
    # Extract the final cutoff score
    print("\nFinal Cutoff Scores for each College:")
    for j in active_colleges:
        final_cutoff = None
        for s in S_j_dict[j]:
            if pyo.value(model.t[j, s]) > 0.5:
                final_cutoff = s
                break
        
        if final_cutoff is not None:
            print(f"College {j:2} | Cutoff Score: {final_cutoff:.4f}")
        else:
            print(f"College {j:2} | No Cutoff (All rejected or 0 applicants)")
            
    print("\nFirst 10 Assignments:")
    print(df_results.head(10))

else:
    print(f"\nSolver did not find an optimal solution. Status: {results.solver.termination_condition}")

Loading and slicing data for Ties (Hungarian Binary Policy)...
Data ready: 1500 Students, 20 Colleges, 11158 Applications.
Total Binary Cutoff Variables (t_j^k) to create: 1011

Building the Pyomo model (SO-H-NW-BIN-CUT)...
Solving the model with GLPK...

Optimal Solution Found in 33.8940 seconds
Total Students Assigned: 1490 out of 1500
Solver Time: 33.8940 seconds
Solution Signature (Hash): -5920545741162952938

Final Cutoff Scores for each College:
College  4 | Cutoff Score: 160.0000
College  3 | Cutoff Score: 30.0000
College 14 | Cutoff Score: 150.0000
College 20 | Cutoff Score: 260.0000
College  9 | Cutoff Score: 210.0000
College 17 | Cutoff Score: 140.0000
College  5 | Cutoff Score: 170.0000
College 12 | Cutoff Score: 210.0000
College 11 | Cutoff Score: 150.0000
College  2 | Cutoff Score: 60.0000
College  6 | Cutoff Score: 150.0000
College 15 | Cutoff Score: 30.0000
College  1 | Cutoff Score: 20.0000
College 18 | Cutoff Score: 60.0000
College 19 | Cutoff Score: 60.0000
College 16